## NVIDIA DLI at ModelScope Notebook

原课程：NVIDIA DLI《深度学习基础》，来源 [https://github.com/NVDLI/fundamentals-of-deep-learning-zh](https://github.com/NVDLI/fundamentals-of-deep-learning-zh) 。本版本由 VoyagerX 适配，原课程正文和练习保留。模型与数据集从 VoyagerX 的 ModelScope 仓库下载，不访问 Hugging Face、Kaggle 或 PyTorch 外部下载站。

## 课程资源最小验证

验证三个数据集、三个预训练模型和离线前向推理。镜像已安装所需依赖，直接按顺序运行；首次从 ModelScope 下载资源，后续使用本地缓存。


In [ ]:
import os, sys
from pathlib import Path
tutorial_dir = Path("/mnt/workspace/fundamentals-of-deep-learning-zh/tutorials")
if not tutorial_dir.is_dir():
    # 首次运行下载课程目录；跳过空链接并打印对应文件名。
    !python -c 'import requests; from pathlib import Path; from concurrent.futures import ThreadPoolExecutor; b="https://modelscope.cn"; gid="cb1503c2-1458-4f67-b9c3-bd505091cb47"; dst=Path("/mnt/workspace/fundamentals-of-deep-learning-zh"); r=requests.get(b+"/api/v1/gallery",params={"Gid":gid},timeout=60); r.raise_for_status(); files=r.json()["Data"]["Gallery"]["Files"]; r=requests.post(b+"/api/v1/gallery/square/files/download",json={"Gid":gid,"FileNames":[f for f in files if f.startswith("course_content/")],"Draft":False},timeout=60); r.raise_for_status(); urls=r.json()["Data"]["Urls"]; exec("def download(x):\n u=x.get(\"Url\")\n if not u:\n  print(\"跳过空下载链接：\",x.get(\"Filename\")); return\n p=dst/x[\"Filename\"].removeprefix(\"course_content/\")\n p.parent.mkdir(parents=True,exist_ok=True)\n r=requests.get(u,timeout=300)\n r.raise_for_status()\n p.write_bytes(r.content)"); pool=ThreadPoolExecutor(max_workers=6); list(pool.map(download,urls)); pool.shutdown(); print("有效链接下载完成：",dst)'
os.chdir(tutorial_dir); sys.path.insert(0, str(tutorial_dir))
from modelscope_assets import prepare_mnist, prepare_asl, prepare_corgi, load_vgg16, prepare_bert


In [ ]:
prepare_mnist()
train = torchvision.datasets.MNIST("data", train=True, download=False)
test = torchvision.datasets.MNIST("data", train=False, download=False)
assert len(train) == 60000 and len(test) == 10000
image, label = train[0]
assert image.size == (28, 28)
print("MNIST:", len(train), len(test), "first label:", label)


In [ ]:
train_df, valid_df = prepare_asl()
assert train_df.shape == (19200, 785) and valid_df.shape == (4800, 785)
assert sorted(train_df["label"].unique().tolist()) == list(range(24))
sample = torch.tensor(train_df.iloc[0, 1:].to_numpy(), dtype=torch.float32).reshape(1, 1, 28, 28) / 255
classifier = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(784, 24))
assert classifier(sample).shape == (1, 24)
print("ASL-HG:", train_df.shape, valid_df.shape, "24-class forward pass OK")
del train_df, valid_df, classifier, sample
gc.collect()


In [ ]:
folder = prepare_corgi()
from PIL import Image
files = [p for p in folder.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
assert len(files) == 202
Image.open(files[0]).verify()
print("Penny the Corgi:", len(files), "images; original class folders retained")


In [ ]:
from torchvision.models import VGG16_Weights
model = load_vgg16().eval()
image = Image.open("data/doggy_door_images/happy_dog.jpg").convert("RGB")
x = VGG16_Weights.DEFAULT.transforms()(image).unsqueeze(0)
with torch.no_grad():
    logits = model(x)
assert logits.shape == (1, 1000)
print("VGG16:", tuple(logits.shape), "predicted class:", int(logits.argmax(1)))
del model, logits, x
gc.collect()


In [ ]:
from transformers import BertTokenizer, BertForMaskedLM
path = prepare_bert("bert-base-cased")
tokenizer = BertTokenizer.from_pretrained(path, local_files_only=True)
model = BertForMaskedLM.from_pretrained(path, local_files_only=True).eval()
inputs = tokenizer("Paris is the [MASK] of France.", return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
assert logits.shape[-1] == 28996
mask_pos = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
pred = tokenizer.decode([int(logits[0, mask_pos].argmax())])
print("BERT masked LM:", tuple(logits.shape), "prediction:", pred)
del model, inputs, logits, tokenizer
gc.collect()


In [ ]:
from transformers import BertTokenizer, BertForQuestionAnswering
path = prepare_bert("bert-large-uncased-whole-word-masking-finetuned-squad")
tokenizer = BertTokenizer.from_pretrained(path, local_files_only=True)
model = BertForQuestionAnswering.from_pretrained(path, local_files_only=True).eval()
inputs = tokenizer("Where do I live?", "My name is Wolfgang and I live in Berlin.", return_tensors="pt")
with torch.no_grad():
    output = model(**inputs)
start = int(output.start_logits.argmax())
end = int(output.end_logits.argmax())
assert output.start_logits.shape == inputs["input_ids"].shape
answer = tokenizer.decode(inputs["input_ids"][0, start:end+1])
print("BERT QA:", tuple(output.start_logits.shape), "answer:", answer)
del model, inputs, output, tokenizer
gc.collect()
print("All asset/model smoke checks passed.")
